In [ ]:
library(plyr)
library(dplyr)
library(tidyr)
library(lme4)
library(lmerTest)
library(interactions)
library(ggplot2)
library(emmeans)
library(rempsyc)
library(ggeffects)

# Feedback count analyses

In [ ]:
# 1 - good job
# 2 - respond faster (pt has too high accuracy >=90%)
# 3 - respond more accurately (pt makes too many mistakes <75%)

rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s/paper_revision/FB", analysis_path, session))
if (!dir.exists(pic_path)) {
  dir.create(pic_path, recursive = TRUE)
}
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/paper_revision/*feedback*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/FB", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}

In [ ]:
# 1. Convert variables to factors
tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$Category <- as.factor(tf_data$Category)

# 2. Revalue 2-level factors for interpretability (if not already run in your environment)
tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))

# 3. Apply sum contrast coding for factors (crucial for interpreting interactions)
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# For Category (3 levels), use contr.sum dynamically based on the number of levels
# contrasts(tf_data$Category) <- contr.sum(length(levels(tf_data$Category)))

# 4. Z-score continuous predictors
tf_data$age_m <- as.numeric(scale(tf_data$age_m))

tf_data$log_Total_Blocks <- log(tf_data$Total_Blocks)

In [ ]:
model <- glmer(Count ~ Category * age_m * soc + sex + dp_inperson + (1 | sub:Category), family = poisson(link = "log"), data = tf_data, control = glmerControl(optimizer = "bobyqa"))
print(summary(model))

In [ ]:
library(emmeans)

# 1. Extract the simple slopes of age_m for each Category and Social Condition
trends_3way <- emtrends(model, ~ soc * Category, var = "age_m")

# 2. Test if each specific slope is significantly different from 0 (is there an age effect in that specific condition?)
print("--- Significance of Simple Slopes (FDR Corrected) ---")
summary(trends_3way, infer = c(TRUE, TRUE), adjust = "fdr")

# 3. Compare the slopes across conditions (e.g., does the age slope for Category 3 differ between NS and S?)
print("--- Pairwise Comparisons of Slopes (FDR Corrected) ---")
# 'simple = "each"' automatically slices the comparisons so you get:
# - Differences between Social Conditions WITHIN each Category
# - Differences between Categories WITHIN each Social Condition
# pairs(trends_3way, simple = "each", adjust = "fdr")

In [ ]:
library(emmeans)

# 1. Extract the age slopes for all conditions
trends_3way <- emtrends(model, ~ soc * Category, var = "age_m")

# 2. Isolate the social effect (S - NS) within each Category
soc_effects <- contrast(trends_3way, "pairwise", by = "Category", reverse = TRUE)

# 3. Test the 'difference of differences' by comparing Categories 2 & 3 to Category 1
diff_of_diffs <- contrast(soc_effects, "trt.vs.ctrl1", by = NULL)

summary(diff_of_diffs, adjust = "none")

In [ ]:
# model_poisson <- glmer(
#   Count ~ Category * age_m * soc + sex + dp_inperson + (1 | sub:Category), 
#   family = poisson(link = "log"), 
#   data = tf_data, 
#   control = glmerControl(optimizer = "bobyqa")
# )

# summary(model_poisson)


# 1. Define the dependent variables you want to iterate over
outcomes <- c(
    'Count'
)

# 2. Define the static right-hand side of your model formula
predictors <- "Category * age_m * soc + sex + dp_inperson + (1 | sub:Category)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- glmer(f, family = poisson(link = "log"), data = tf_data, control = glmerControl(optimizer = "bobyqa"))
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms_full")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}

In [ ]:
confint(model)

In [ ]:
library(dplyr)
library(tidyr)

# 1. How many participants are missing each category?
missing_summary <- tf_data %>%
  group_by(sub, Category) %>%
  summarize(total_count = sum(Count, na.rm = TRUE), .groups = "drop") %>%
  complete(sub, Category, fill = list(total_count = 0)) %>%
  group_by(Category) %>%
  summarize(participants_missing = sum(total_count == 0))

print(missing_summary)

In [ ]:
# 2. Get a list of specific participant IDs missing each category
missing_subjects <- tf_data %>%
  group_by(sub, Category) %>%
  summarize(total_count = sum(Count, na.rm = TRUE), .groups = "drop") %>%
  complete(sub, Category, fill = list(total_count = 0)) %>%
  filter(total_count == 0) %>%
  arrange(Category, sub)

print(missing_subjects)

In [ ]:
library(dplyr)
library(tidyr)

# 1. Calculate the zeros per Category AND Social condition (soc)
zero_summary_soc <- tf_data %>%
  group_by(sub, Category, soc) %>%
  summarize(total_count = sum(Count, na.rm = TRUE), .groups = "drop") %>%
  # Complete now ensures every sub has every Category * soc combination
  complete(sub, Category, soc, fill = list(total_count = 0)) %>%
  group_by(Category, soc) %>%
  summarize(zero_people = sum(total_count == 0), .groups = "drop") %>%
  arrange(Category, soc)

# 2. Print in text format
for (i in 1:nrow(zero_summary_soc)) {
  cat(sprintf("zero of cat %s in %s: %d people\n", 
              zero_summary_soc$Category[i], 
              zero_summary_soc$soc[i], 
              zero_summary_soc$zero_people[i]))
}

In [ ]:
library(ggplot2)
library(dplyr)

# Option 1: Boxplot with jittered raw data (Best for showing the zero-counts and distributions)
ggplot(tf_data, aes(x = as.factor(Category), y = Count, fill = soc)) +
  geom_boxplot(position = position_dodge(width = 0.8), outlier.shape = NA, alpha = 0.7) +
  geom_point(position = position_jitterdodge(jitter.width = 0.2, dodge.width = 0.8), 
             alpha = 0.3, size = 1) +
  labs(
    title = "Distribution of Feedback Counts by Category and Social Condition",
    x = "Feedback Category",
    y = "Number of Feedbacks Received",
    fill = "Social Condition"
  ) +
  scale_fill_manual(values = c("NS" = "#43a2ca", "S" = "#e34a33"),
                    labels = c("NS" = "Alone (NS)", "S" = "Observed (S)")) +
  theme_classic(base_size = 14)


# Option 2: Summarized Bar Plot with Standard Error (Classic manuscript style)
tf_summary <- tf_data %>%
  group_by(soc, Category) %>%
  summarize(
    mean_count = mean(Count, na.rm = TRUE),
    se_count = sd(Count, na.rm = TRUE) / sqrt(n()),
    .groups = "drop"
  )

ggplot(tf_summary, aes(x = as.factor(Category), y = mean_count, fill = soc)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7, color = "black") +
  geom_errorbar(aes(ymin = mean_count - se_count, ymax = mean_count + se_count),
                position = position_dodge(width = 0.8), width = 0.25) +
  labs(
    title = "Mean Feedback Counts by Category and Social Condition",
    x = "Feedback Category",
    y = "Mean Number of Feedbacks",
    fill = "Social Condition"
  ) +
  scale_fill_manual(values = c("NS" = "#43a2ca", "S" = "#e34a33"),
                    labels = c("NS" = "Alone (NS)", "S" = "Observed (S)")) +
  theme_classic(base_size = 14)

In [ ]:
library(dplyr)

# Count the number of observations (rows) per participant per condition
observation_check <- tf_data %>%
  group_by(sub, soc) %>%
  summarize(n_obs = n(), .groups = "drop") %>%
  filter(n_obs != 3)

# Print results
if (nrow(observation_check) == 0) {
  print("Success: Every participant has exactly 3 observations in each condition.")
} else {
  print("Warning: The following participants/conditions do not have exactly 3 observations:")
  print(observation_check)
}

In [ ]:
model_poisson <- glmer(
  Count ~ Category + Category:(age_m * soc) + sex + dp_inperson + (1 | sub:Category), 
  family = poisson(link = "log"), 
  data = tf_data, 
  control = glmerControl(optimizer = "bobyqa")
)

summary(model_poisson)

In [ ]:
model_poisson <- glmer(
  Count ~ Category + Category:(age_m * soc) + Category:(sex) + Category:(dp_inperson) + (1 | sub:Category), 
  family = poisson(link = "log"), 
  data = tf_data, 
  control = glmerControl(optimizer = "bobyqa")
)

summary(model_poisson)

In [ ]:
model_poisson <- glmer(
 Count ~ Category * (age_m * soc + sex + dp_inperson) + (1 | sub:Category),
  family = poisson(link = "log"), 
  data = tf_data, 
  control = glmerControl(optimizer = "bobyqa")
)

summary(model_poisson)

In [ ]:
library(emmeans)

# 1. Extract the simple slopes of age_m for each Category and Social Condition
trends_3way <- emtrends(model, ~ soc * Category, var = "age_m")

# 2. Test if each specific slope is significantly different from 0 (is there an age effect in that specific condition?)
print("--- Significance of Simple Slopes (FDR Corrected) ---")
summary(trends_3way, infer = c(TRUE, TRUE), adjust = "fdr")

# 3. Compare the slopes across conditions (e.g., does the age slope for Category 3 differ between NS and S?)
print("--- Pairwise Comparisons of Slopes (FDR Corrected) ---")
# 'simple = "each"' automatically slices the comparisons so you get:
# - Differences between Social Conditions WITHIN each Category
# - Differences between Categories WITHIN each Social Condition
# pairs(trends_3way, simple = "each", adjust = "fdr")

In [ ]:
# 1. Generate predicted values across a continuous sequence of age_m
# (This uses the built-in emmip function from emmeans to create the plot data)
plot_lines <- emmip(model_poisson, soc ~ age_m | Category, 
                    cov.reduce = range, plotit = FALSE, CIs = TRUE)

# 2. Plot with ggplot2
ggplot(plot_lines, aes(x = age_m, y = yvar, color = soc, fill = soc)) +
  geom_line(size = 1.2) +
  geom_ribbon(aes(ymin = LCL, ymax = UCL), alpha = 0.2, color = NA) +
  facet_wrap(~ Category, labeller = label_both) +
  labs(
    title = "Predicted Feedback Counts Across Age",
    x = "Age (mean-centered)",
    y = "Predicted Log-Count",
    color = "Social Condition",
    fill = "Social Condition"
  ) +
  scale_color_manual(values = c("NS" = "#43a2ca", "S" = "#e34a33"),
                     labels = c("NS" = "Alone", "S" = "Observed")) +
  scale_fill_manual(values = c("NS" = "#43a2ca", "S" = "#e34a33"),
                    labels = c("NS" = "Alone", "S" = "Observed")) +
  theme_bw(base_size = 14) +
  theme(strip.background = element_rect(fill = "white"))

In [ ]:
library(emmeans)

# 1. Compare the 'soc' conditions (NS vs S) WITHIN each Category 
# The 'by = "Category"' argument forces emmeans to isolate the comparisons per category.
# 'reverse = TRUE' flips the subtraction to S - NS instead of the alphabetical NS - S.
trends_3way <- emtrends(model, ~ soc * Category, var = "age_m")

slope_comparisons <- pairs(trends_3way, by = "Category", reverse = TRUE, adjust = "fdr")

# 2. Extract and print only the row for Category 3
subset(slope_comparisons, Category == "3")

In [ ]:
library(emmeans)

# 1. Extract the age slopes for all conditions
trends_3way <- emtrends(model, ~ soc * Category, var = "age_m")

# 2. Isolate the social effect (S - NS) within each Category
soc_effects <- contrast(trends_3way, "pairwise", by = "Category", reverse = TRUE)

# 3. Test the 'difference of differences' by comparing Categories 2 & 3 to Category 1
diff_of_diffs <- contrast(soc_effects, "trt.vs.ctrl1", by = NULL)

summary(diff_of_diffs, adjust = "none")

In [ ]:
# The '| Category' syntax tells emtrends to nest the calculation within Category
trends_nested <- emtrends(model_poisson, ~ soc | Category, var = "age_m")

# This will automatically give you the S - NS comparisons within each Category
pairs(trends_nested, reverse = TRUE, adjust = "fdr")

In [ ]:
# 5. Run the Poisson Mixed Effects Model
# model_poisson <- glmer(
#   Count ~ age_m * soc * Category + sex:Category + dp_inperson:Category + (1 | Category:sub) + offset(log_Total_Blocks), 
#   family = poisson(link = "log"), 
#   data = tf_data, 
#   control = glmerControl(optimizer = "bobyqa")
# )



# model_poisson <- glmer(Count ~ age_m * soc * Category + sex:Category + dp_inperson:Category + (0 + Category | sub), 
#                          family = poisson(link = "log"), 
#   data = tf_data, 
#   control = glmerControl(optimizer = "bobyqa")
# )
summary(model_poisson)

# model_poisson <- glm(Count ~ age_m * sex * soc * Category + offset(log(Total_Blocks)), 
#                  family = poisson(link = "log"), 
#                  data = tf_data)

# summary(model_poisson)

# model_poisson <- glm(Count ~ age_m * soc * Category + sex + dp_inperson,
#                      offset(log_Total_Blocks), 
#                      family = poisson(link = "log"), 
#                      data = tf_data)

# summary(model_poisson)

# zero inflated models

In [ ]:
unique(tf_data$Total_Blocks)

In [ ]:
library(emmeans)

# 1. Calculate the simple slopes (trends) of age_m for each combination of soc and Category
trends_3way <- emtrends(model_poisson, ~ soc * Category, var = "age_m")

# 2. Test if the slope of age_m in each specific condition is significantly different from 0
print("--- Significance of each simple slope from 0 ---")
test(trends_3way, adjust = "fdr")

# 3. Pairwise comparisons of the slopes between conditions
print("--- Pairwise comparisons of age_m slopes ---")
# simple = "each" will automatically test 'soc' differences within 'Category', 
# and 'Category' differences within 'soc'
pairs(trends_3way, simple = "each", adjust = "fdr")

In [ ]:
# ==============================================================================
# 1. 2-Way Interaction: sex x Category (sex1:Category2 / sex1:Category3)
# ==============================================================================

# Follow-up Comparisons: Estimated Marginal Means (EMMs)
# type = "response" back-transforms the log-link to the raw count scale
emm_2way <- emmeans(model_poisson, ~ sex * Category, type = "response")

# Compare 'sex' differences within each 'Category'
cat("--- Pairwise comparisons of Sex within each Category ---\n")
pairs(emm_2way, simple = "sex", adjust='fdr')

# Compare 'Category' differences within each 'Sex'
cat("\n--- Pairwise comparisons of Categories within each Sex ---\n")
pairs(emm_2way, simple = "Category", adjust='fdr')

# Plot the 2-way interaction
plot_2way <- ggpredict(model_poisson, terms = c("Category", "sex"))
plot(plot_2way) + 
  labs(title = "Predicted Counts: Sex by Category", 
       y = "Predicted Count", 
       x = "Category", 
       color = "Sex") +
  theme_minimal()

# ==============================================================================
# 2. 4-Way Interaction: age_m x sex x soc x Category 
# ==============================================================================

# Follow-up Comparisons: Simple Slopes Analysis
# This calculates the slope (log-incidence rate ratio) of age_m for every combination
trends_4way <- emtrends(model_poisson, ~ sex * soc * Category, var = "age_m")

# 1. Test if each specific simple slope is significantly different from 0
cat("--- Simple slopes of age_m for each combination of Sex, Soc, and Category ---\n")
test(trends_4way, adjust='fdr')

# 2. Compare the slopes across groups to find what drives the interaction
# (e.g., does the age slope differ between males and females within soc 'S' and Category 3?)
cat("\n--- Pairwise comparisons of age_m slopes ---\n")
pairs(trends_4way, adjust='fdr')

# Plot the 4-way interaction
# ggpredict handles multi-way continuous x categorical interactions exceptionally well
# terms mapping: x-axis = age_m, color/lines = sex, facets = soc, panels = Category
# plot_4way <- ggpredict(model_poisson, terms = c("age_m", "sex", "soc", "Category"))

# Swap 'soc' to the 2nd position (color) and 'sex' to the 3rd (facet)
plot_4way_flipped <- ggpredict(model_poisson, terms = c("age_m", "soc", "sex", "Category"))

plot(plot_4way_flipped, facets = TRUE) +
  labs(title = "4-Way Interaction: Age x Condition (soc) x Sex x Category",
       y = "Predicted Count", 
       x = "Age (Z-scored)", 
       color = "Condition") + # Updated legend title to match 'soc'
  theme_minimal() +
  theme(strip.text = element_text(size = 10, face = "bold"))

# 16-trial cutoff sensitivity analyses

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/paper_revision/*sensitivity*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}

In [ ]:
library(dplyr)

# 1. Extract one unique row per subject
df_demo <- tf_data %>% 
  distinct(sub, .keep_all = TRUE)

# Define the conditions to iterate over
conditions <- c(0, 1) 

for (cond in conditions) {
  cat("\n========================================================\n")
  cat("   RESULTS FOR CONDITION:", toupper(cond), "\n")
  cat("========================================================\n")

    df_sub <- tf_data %>% 
    filter(soc == cond) %>% 
    distinct(sub, .keep_all = TRUE)
  
  # Safety check: Ensure both groups exist in the subset to prevent t.test errors
  if(length(unique(df_sub$group)) < 2) {
    cat("Error: Less than 2 groups present in this condition. Skipping...\n")
    next
  }
  
  # 2. Compare Age between groups
  cat("\n--- Independent Samples T-Test (Age) ---\n")
  t_test_age <- t.test(age_m ~ group, data = df_sub)
  print(t_test_age)
  
  # 3. Compare Sex distribution between groups
  cat("\n--- Chi-Squared Test (Sex) ---\n")
  sex_table <- table(df_sub$sex, df_sub$group)
  print(sex_table)
    
  # significantly increases the chance of cell counts falling below 5.
  chisq_sex <- chisq.test(sex_table)
  print(chisq_sex)
}

In [ ]:
library(dplyr)
library(knitr)
tf_data$age_m = tf_data$age_m / 12
# Initialize an empty list to store the results of each iteration
results_list <- list()
conditions <- c(0, 1) 

for (cond in conditions) {
  
  df_sub <- tf_data %>% 
    filter(soc == cond) %>% 
    distinct(sub, .keep_all = TRUE)
  
  # Safety check
  if(length(unique(df_sub$group)) < 2) {
    next
  }
  
  # 1. Calculate Age Means and SDs
  age_summary <- df_sub %>%
    group_by(group) %>%
    summarise(
      mean = mean(age_m, na.rm = TRUE),
      sd = sd(age_m, na.rm = TRUE),
      .groups = 'drop'
    )
  
  # Helper function to format Mean and SD securely
  get_age_stat <- function(g) {
    val <- age_summary[age_summary$group == g, ]
    if(nrow(val) > 0) sprintf("%.2f ± %.2f", val$mean, val$sd) else NA
  }
  
  # 2. Perform Age T-Test
  t_test_age <- t.test(age_m ~ group, data = df_sub)
  
  # 3. Create Sex table and perform Chi-Squared Test
  sex_table <- table(df_sub$sex, df_sub$group)
  suppressWarnings(chisq_sex <- chisq.test(sex_table))
  
  # Helper function to safely extract sex counts preventing out-of-bounds errors
  get_sex_count <- function(s, g) {
    if (as.character(s) %in% rownames(sex_table) && g %in% colnames(sex_table)) {
      return(sex_table[as.character(s), g])
    } else {
      return(0)
    }
  }
  
  # 4. Bind all extracted data into a single row for this condition
  results_list[[length(results_list) + 1]] <- data.frame(
    Condition = cond,
    Age_FEW_ERR_Mean_SD = get_age_stat("FEW_ERR"),
    Age_MANY_ERR_Mean_SD = get_age_stat("MANY_ERR"),
    Age_P_Value = round(t_test_age$p.value, 4),
    Sex_1_FEW_ERR = get_sex_count("1", "FEW_ERR"),
    Sex_2_FEW_ERR = get_sex_count("2", "FEW_ERR"),
    Sex_1_MANY_ERR = get_sex_count("1", "MANY_ERR"),
    Sex_2_MANY_ERR = get_sex_count("2", "MANY_ERR"),
    Sex_P_Value = round(chisq_sex$p.value, 4)
  )
}

# Combine everything into a final table and print cleanly using kable
final_table <- bind_rows(results_list)
kable(final_table, format = "markdown", align = "c")

In [ ]:
final_table

In [ ]:
nrow(subset(df_demo, df_demo$group == 'FEW_ERR'))

In [ ]:
nrow(subset(tf_data, tf_data$group == 'FEW_ERR'))

In [ ]:
nrow(tf_data[tf_data$group == 'FEW_ERR' & tf_data$soc == 1 & tf_data$sex == 1, ])

In [ ]:
nrow(tf_data[tf_data$group == 'FEW_ERR' & tf_data$acc == 1 & tf_data$soc == 0, ])

In [ ]:
nrow(subset(df_demo, df_demo$group == 'MANY_ERR'))

In [ ]:
library(dplyr)
library(tidyr)

# 1. Define the variables that must ALL be valid
subset_vars <- c(
  "pes",
  "peri_rt",
  "pea",
  "peri_acc",
  "reversed_ratio_diff",
  "a_diff"
)

# 2. Run the demographic comparisons per condition
for (s_cond in c(1, 0)) {
  
  # Subset to condition, drop rows if ANY of the subset_vars are NA, and keep 1 row per subject
  df_sub <- tf_data %>%
    filter(soc == s_cond) %>%
    drop_na(all_of(subset_vars)) %>%
    group_by(sub, group) %>%
    summarize(
      age_m = first(age_m),
      sex = first(sex),
      .groups = "drop"
    )
  
  # Count how many valid subjects are left in each group
  n_few <- sum(df_sub$group == "FEW_ERR")
  n_many <- sum(df_sub$group == "MANY_ERR")
  
  # Print clean headers and sample sizes
  cat(sprintf("\n======================================================\n"))
  cat(sprintf("Subset valid for ALL columns | Condition (soc): %s\n", s_cond))
  cat(sprintf("Valid Subjects -> FEW_ERR: %d | MANY_ERR: %d\n", n_few, n_many))
  cat(sprintf("------------------------------------------------------\n"))
  
  if (n_few >= 2 && n_many >= 2) {
    
    # --- Compare age_m (t-test) ---
    cat(">>> Comparing age_m:\n")
    print(t.test(age_m ~ group, data = df_sub))
    
    # --- Compare sex (Fisher's Exact Test) ---
    cat(">>> Comparing sex:\n")
    sex_tab <- table(df_sub$group, df_sub$sex)
    print(sex_tab) # Show the actual counts
    
    tryCatch({
      print(fisher.test(sex_tab))
    }, error = function(e) {
      cat("Could not run Fisher's test.\n")
    })
    
  } else {
    cat("Not enough valid subjects in one or both groups to run statistical tests.\n")
  }
}

In [ ]:
library(dplyr)
library(tidyr)

# 1. Define the variables
subset_vars <- c(
  "pes",
  "peri_rt",
  "pea",
  "peri_acc",
  "reversed_ratio_diff",
  "a_diff"
)

# 2. Run the demographic comparisons per condition
for (s_cond in c(1, 0)) {
  
  # Subset to condition, keep rows if AT LEAST ONE of the subset_vars is not NA, and keep 1 row per subject
  df_sub <- tf_data %>%
    filter(soc == s_cond) %>%
    filter(if_any(all_of(subset_vars), ~ !is.na(.))) %>%
    group_by(sub, group) %>%
    summarize(
      age_m = first(age_m),
      sex = first(sex),
      .groups = "drop"
    )
  
  # Count how many valid subjects are left in each group
  n_few <- sum(df_sub$group == "FEW_ERR")
  n_many <- sum(df_sub$group == "MANY_ERR")
  
  # Print clean headers and sample sizes
  cat(sprintf("\n======================================================\n"))
  cat(sprintf("Subset valid for AT LEAST ONE column | Condition (soc): %s\n", s_cond))
  cat(sprintf("Valid Subjects -> FEW_ERR: %d | MANY_ERR: %d\n", n_few, n_many))
  cat(sprintf("------------------------------------------------------\n"))
  
  if (n_few >= 2 && n_many >= 2) {
    
    # --- Compare age_m (t-test) ---
    cat(">>> Comparing age_m:\n")
    print(t.test(age_m ~ group, data = df_sub))
    
    # --- Compare sex (Fisher's Exact Test) ---
    cat(">>> Comparing sex:\n")
    sex_tab <- table(df_sub$group, df_sub$sex)
    print(sex_tab) # Show the actual counts
    
    tryCatch({
      print(fisher.test(sex_tab))
    }, error = function(e) {
      cat("Could not run Fisher's test.\n")
    })
    
  } else {
    cat("Not enough valid subjects in one or both groups to run statistical tests.\n")
  }
}

In [ ]:
library(dplyr)

# 1. Define the variables used to define the valid subsets
subset_vars <- c(
"pes",
  "peri_rt",
  "pea",
  "peri_acc",
  "reversed_ratio_diff",
  "a_diff"
)

# 2. Run the 6x2 = 12 demographic comparisons
for (v in subset_vars) {
  # Using 1 and 0 for soc as seen in your previous cell output
  for (s_cond in c(1, 0)) {
    
    # Subset to the condition, drop NAs for this variable, and keep 1 row per subject
    df_sub <- tf_data %>%
      filter(soc == s_cond, !is.na(.data[[v]])) %>%
      group_by(sub, group) %>%
      summarize(
        age_m = first(age_m),
        sex = first(sex),
        .groups = "drop"
      )
    
    # Count how many valid subjects are left in each group
    n_few <- sum(df_sub$group == "FEW_ERR")
    n_many <- sum(df_sub$group == "MANY_ERR")
    
    # Print clean headers and sample sizes
    cat(sprintf("\n======================================================\n"))
    cat(sprintf("Subset valid for: %s | Condition (soc): %s\n", v, s_cond))
    cat(sprintf("Valid Subjects -> FEW_ERR: %d | MANY_ERR: %d\n", n_few, n_many))
    cat(sprintf("------------------------------------------------------\n"))
    
    if (n_few >= 2 && n_many >= 2) {
      
      # --- Compare age_m (t-test) ---
      cat(">>> Comparing age_m:\n")
      print(t.test(age_m ~ group, data = df_sub))
      
      # --- Compare sex (Fisher's Exact Test) ---
      cat(">>> Comparing sex:\n")
      sex_tab <- table(df_sub$group, df_sub$sex)
      print(sex_tab) # Show the actual counts
      
      # Use tryCatch in case a subset has only one sex present, which breaks the test
      tryCatch({
        print(fisher.test(sex_tab))
      }, error = function(e) {
        cat("Could not run Fisher's test (possibly only one sex present in this subset).\n")
      })
      
    } else {
      cat("Not enough valid subjects in one or both groups to run statistical tests.\n")
    }
  }
}

In [ ]:
library(dplyr)

# 1. Define the variables to loop through
vars_to_test <- c(
  "pes",
  "peri_rt",
  "pea",
  "peri_acc",
  "reversed_ratio_diff",
  "a_diff"
)

# 2. Run the 6x2 = 12 comparisons
for (v in vars_to_test) {
  for (s_cond in c(1, 0)) {
    
    # Subset to the condition, drop NAs for this variable, and aggregate to 1 row per subject
    df_sub <- tf_data %>%
      filter(soc == s_cond, !is.na(.data[[v]])) %>%
      group_by(sub, group) %>%
      summarize(subject_mean = mean(.data[[v]], na.rm = TRUE), .groups = "drop")
    
    # Count how many valid subjects are left in each group
    n_few <- sum(df_sub$group == "FEW_ERR")
    n_many <- sum(df_sub$group == "MANY_ERR")
    
    # Print clean headers and sample sizes
    cat(sprintf("\n======================================================\n"))
    cat(sprintf("Variable: %s | Condition: %s\n", v, s_cond))
    cat(sprintf("Valid Subjects -> FEW_ERR: %d | MANY_ERR: %d\n", n_few, n_many))
    cat(sprintf("------------------------------------------------------\n"))
    
    # Run the Welch Two Sample t-test if both groups have enough data
    if (n_few >= 2 && n_many >= 2) {
      ttest_res <- t.test(subject_mean ~ group, data = df_sub)
      print(ttest_res)
    } else {
      cat("Not enough valid subjects in one or both groups to run a t-test.\n")
    }
  }
}

In [ ]:
  "rt_log_post_trial",
    "rt_interference_post_trial",
    "acc_interference_post_trial",
  "accuracy_score_post_trial",
  "reversed_ratio_post_trial",
  "a_post_trial"

In [ ]:
nrow(df_demo)

In [ ]:
head(tf_data)

In [ ]:
# 1. Convert variables to factors
tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)
tf_data$group <- as.factor(tf_data$group)

# 2. Revalue 2-level factors for interpretability (if not already run in your environment
tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

# 3. Apply sum contrast coding for factors (crucial for interpreting interactions)
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))
contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$group) <- rev(contr.sum(2))
# For Category (3 levels), use contr.sum dynamically based on the number of levels
# contrasts(tf_data$Category) <- contr.sum(length(levels(tf_data$Category)))

# 4. Z-score continuous predictors
tf_data$age_m <- scale(tf_data$age_m)
tf_data$ICPS_early_DLPFC_collapsed <- scale(tf_data$ICPS_early_DLPFC_collapsed)
tf_data$ICPS_early_MOTOR_collapsed <- scale(tf_data$ICPS_early_MOTOR_collapsed)
tf_data$ICPS_early_OCC_collapsed <- scale(tf_data$ICPS_early_OCC_collapsed)

In [ ]:
m_dlpfc <- lmer(ICPS_early_DLPFC_collapsed ~ soc * age_m * acc * group + (1 | sub), data = tf_data)
summary(m_dlpfc)

m_motor <- lmer(ICPS_early_MOTOR_collapsed ~ soc * age_m * acc * group + (1 | sub), data = tf_data)
summary(m_motor)

m_occ <- lmer(ICPS_early_OCC_collapsed ~ soc * age_m * acc * group + (1 | sub), data = tf_data)
summary(m_occ)

In [ ]:
library(emmeans)
library(ggplot2)

# ==============================================================================
# 1. DLPFC MODEL
# Significant group interaction: acc:group (p = .018)
# ==============================================================================

# A. Full pairwise comparisons across all 4 cells of acc x group
dlpfc_acc_group_all <- emmeans(m_dlpfc, pairwise ~ acc * group, adjust = "fdr")
cat("\n--- DLPFC: Full Pairwise Comparisons (acc x group) ---\n")
print(dlpfc_acc_group_all)

# B. Simple main effects (often more interpretable)
cat("\n--- DLPFC: Accuracy differences within each Group ---\n")
print(emmeans(m_dlpfc, pairwise ~ acc | group, adjust = "fdr"))

cat("\n--- DLPFC: Group differences within each Accuracy level ---\n")
print(emmeans(m_dlpfc, pairwise ~ group | acc, adjust = "fdr"))

# Plot
emmip(m_dlpfc, acc ~ group) + 
  theme_minimal() + 
  labs(title = "DLPFC: Accuracy by Group", x = "Group", y = "Estimated ICPS")


# ==============================================================================
# 2. OCC MODEL
# Significant group interactions: 
# acc:group (p = .004), soc:age_m:group (p = .002), soc:age_m:acc:group (p = .025)
# *Note: The 4-way interaction supersedes the lower-order ones, but all are provided.
# ==============================================================================

# ---------------------------------------------------------
# A. 2-Way: acc:group
# ---------------------------------------------------------
cat("\n--- OCC 2-WAY: Full Pairwise Comparisons (acc x group) ---\n")
print(emmeans(m_occ, pairwise ~ acc * group, adjust = "fdr"))

cat("\n--- OCC 2-WAY: Accuracy within Group & Group within Accuracy ---\n")
print(emmeans(m_occ, pairwise ~ acc | group, adjust = "fdr"))
print(emmeans(m_occ, pairwise ~ group | acc, adjust = "fdr"))

# ---------------------------------------------------------
# B. 3-Way: soc:age_m:group
# Tests differences in the continuous age_m slope
# ---------------------------------------------------------
cat("\n--- OCC 3-WAY: Full Pairwise Age Slopes (soc x group) ---\n")
# Compares age slopes among all 4 combinations of soc and group
print(emtrends(m_occ, pairwise ~ soc * group, var = "age_m", adjust = "fdr"))

cat("\n--- OCC 3-WAY: Simple Slopes (soc within group & group within soc) ---\n")
print(emtrends(m_occ, pairwise ~ soc | group, var = "age_m", adjust = "fdr"))
print(emtrends(m_occ, pairwise ~ group | soc, var = "age_m", adjust = "fdr"))

# ---------------------------------------------------------
# C. 4-Way: soc:age_m:acc:group 
# Tests differences in the continuous age_m slope
# ---------------------------------------------------------
cat("\n--- OCC 4-WAY: Full Pairwise Age Slopes (soc x acc x group) ---\n")
# Compares age slopes among all 8 combinations of soc, acc, and group
print(emtrends(m_occ, pairwise ~ soc * acc * group, var = "age_m", adjust = "fdr"))

cat("\n--- OCC 4-WAY: Simple Slopes (soc differences within acc*group) ---\n")
print(emtrends(m_occ, pairwise ~ soc | acc * group, var = "age_m", adjust = "fdr"))

cat("\n--- OCC 4-WAY: Simple Slopes (acc differences within soc*group) ---\n")
print(emtrends(m_occ, pairwise ~ acc | soc * group, var = "age_m", adjust = "fdr"))

# Plot 4-Way Slopes
emmip(m_occ, soc ~ age_m | acc * group, cov.reduce = range) + 
  theme_minimal() + 
  labs(title = "OCC: 4-Way Interaction (soc x age_m x acc x group)", 
       x = "Age (Z-scored)", y = "Estimated ICPS")

# Post-error adjustments effects of accuracy

In [ ]:
rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s/paper_revision/", analysis_path, session))
if (!dir.exists(pic_path)) {
  dir.create(pic_path, recursive = TRUE)
}
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/*long*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/paper_revision", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
o}

In [ ]:
# Install missing packages if necessary: 
# install.packages(c("dplyr", "tidyr", "flextable", "officer"))

library(dplyr)
library(tidyr)
library(flextable)
library(officer)

# 1. Define your variables
# NOTE: Update the right-hand side strings to match the exact column names in your tf_data
vars <- c(
  "Proportion Correct"    = "accuracy_score_post_trial", 
  "Accuracy Interference" = "acc_interference_post_trial",
  "RT Overall (ms)"       = "rt_post_trial",             
  "RT Congruent (ms)"     = "rt_con_post_trial",         
  "RT Incongruent (ms)"   = "rt_incon_post_trial",       
  "RT Interference"       = "rt_interference_post_trial",
    "a Response Caution" = "a_post_trial",
    "sda/rd Attentional Ratio" = "reversed_ratio_post_trial"
)
tf_data$rt_post_trial = tf_data$rt_post_trial*1000
tf_data$rt_con_post_trial = tf_data$rt_con_post_trial*1000
tf_data$rt_incon_post_trial = tf_data$rt_incon_post_trial*1000
summary_df <- tf_data %>%
  # 1. Compute RT interference as Incongruent minus Congruent
  mutate(rt_interference_post_trial = rt_incon_post_trial - rt_con_post_trial) %>%
  
  # 2. Proceed with the rest of the pipeline exactly as before
  select(sub, soc, acc, all_of(vars)) %>%
  pivot_longer(cols = all_of(names(vars)), names_to = "measure", values_to = "value") %>%
  group_by(measure, soc, acc) %>%
  dplyr::summarise(
    M = mean(value, na.rm = TRUE),
    SD = sd(value, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    formatted_val = sprintf("%.2f \u00B1 %.2f", M, SD),
    condition = paste(soc, acc, sep = "_")
  ) %>%
  select(measure, condition, formatted_val) %>%
  pivot_wider(names_from = condition, values_from = formatted_val) %>%
  select(
    `Data type (measure)` = measure,
    `NS Post-Err.`        = `0_0`,  
    `NS Post-Corr.`       = `0_1`,  
    `S Post-Err.`         = `1_0`,  
    `S Post-Corr.`        = `1_1`   
  ) %>%
  arrange(match(`Data type (measure)`, names(vars)))

# Print to console
print(summary_df)

# Format and export
ft <- flextable(summary_df) %>%
  add_header_row(
    values = c("", "Raw Behavioral Measures (M \u00B1 SD)"),
    colwidths = c(1, 4),
    top = FALSE
  ) %>%
  theme_apa() %>%
  align(j = 2:5, align = "center", part = "all") %>%
  add_footer_lines("Note. \"S\" denotes social condition, and \"NS\" denotes nonsocial condition.") %>%
  
  # --- New Formatting Block Applied ---
  set_table_properties(width = 1, layout = "autofit") %>%
  line_spacing(space = 1, part = "all") %>%
  padding(padding.top = 0.5, padding.bottom = 0.5, part = "all") %>%
  fontsize(size = 10, part = "all") %>%
  fontsize(size = 9, part = "footer")

# 5. Export to a Word document (.docx)
save_as_docx(ft, path = sprintf("%s/Behavioral_Measures_Table.docx", table_output_path))

In [ ]:
summary_df

In [ ]:
tf_data %>% 
  filter(soc == 1, acc == 1) %>% 
  summarise(mean_rt = mean(rt_incon_post_trial, na.rm = TRUE))

tf_data %>% 
  filter(soc == 1, acc == 0) %>% 
  summarise(mean_rt = mean(rt_incon_post_trial, na.rm = TRUE))

tf_data %>% 
  filter(soc == 0, acc == 1) %>% 
  summarise(mean_rt = mean(rt_incon_post_trial, na.rm = TRUE))

tf_data %>% 
  filter(soc == 0, acc == 0) %>% 
  summarise(mean_rt = mean(rt_incon_post_trial, na.rm = TRUE))

In [ ]:
# 1. Convert variables to factors
tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

# 2. Revalue 2-level factors for interpretability (if not already run in your environment
tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

# 3. Apply sum contrast coding for factors (crucial for interpreting interactions)
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))
contrasts(tf_data$acc) <- rev(contr.sum(2))
# For Category (3 levels), use contr.sum dynamically based on the number of levels
# contrasts(tf_data$Category) <- contr.sum(length(levels(tf_data$Category)))

# 4. Z-score continuous predictors
tf_data$age_m <- scale(tf_data$age_m)
tf_data = subset(tf_data, tf_data$acc == "Error")
# tf_data$power_early <- scale(tf_data$power_early)
# tf_data$rt_con_log_post_trial <- scale(tf_data$rt_con_log_post_trial)
# tf_data$rt_incon_log_post_trial <- scale(tf_data$rt_incon_log_post_trial)
tf_data$acc_interference_post_trial <- scale(tf_data$acc_interference_post_trial)
tf_data$rt_interference_post_trial <- scale(tf_data$rt_interference_post_trial)
tf_data$rt_log_post_trial <- scale(tf_data$rt_log_post_trial)
tf_data$reversed_ratio_post_trial <- scale(tf_data$reversed_ratio_post_trial)
tf_data$a_post_trial <- scale(tf_data$a_post_trial)
tf_data$ICPS_early_OCC_diff_collapsed = scale(tf_data$ICPS_early_OCC_diff_collapsed)
tf_data$reversed_ratio_diff = scale(tf_data$reversed_ratio_diff)

In [ ]:
library(dplyr)
library(tidyr)

# 1. Create a summary dataframe with the computed difference scores
df_diff_acc <- tf_data %>%
  # Group by subject, social condition, and previous accuracy
  group_by(sub, soc, acc) %>%
  # Calculate mean post-trial accuracy, removing any NAs
  summarise(mean_post_acc = mean(accuracy_score_post_trial, na.rm = TRUE), .groups = "drop") %>%
  # Pivot to wide format: creates columns named 'acc_Error' and 'acc_Correct'
  pivot_wider(names_from = acc, values_from = mean_post_acc, names_prefix = "acc_") %>%
  # Calculate the difference: Post-Error (acc=0) minus Post-Correct (acc=1)
  mutate(PE_diff_accuracy = acc_Error - acc_Correct)

# View the exploratory summary dataframe
head(df_diff_acc)

# 2. (Optional) Merge this difference score back into your long-format data
# This assigns the exact same PE_diff_accuracy value to every trial for a given sub & soc
tf_data <- tf_data %>%
  left_join(
    df_diff_acc %>% select(sub, soc, PE_diff_accuracy),
    by = c("sub", "soc")
  )

In [ ]:
model = lmer(reversed_ratio_diff ~ ICPS_early_DLPFC_diff_collapsed * soc * age_m + ICPS_early_MOTOR_diff_collapsed * soc * age_m + ICPS_early_OCC_diff_collapsed * soc * age_m + sex + dp_inperson + (1 | sub),
             data = tf_data)
summary(model)

In [ ]:
library(lme4)
library(lmerTest)
library(ggplot2)

# 1. Extract fixed effects for the intercept and main effect of ICPS
betas <- fixef(model)
b0 <- betas["(Intercept)"]
b1 <- betas["ICPS_early_OCC_diff_collapsed"] 

# 2. Extract variances and covariance
cov_mat <- vcov(model)
var_b0 <- cov_mat["(Intercept)", "(Intercept)"]
var_b1 <- cov_mat["ICPS_early_OCC_diff_collapsed", "ICPS_early_OCC_diff_collapsed"]
cov_b0_b1 <- cov_mat["(Intercept)", "ICPS_early_OCC_diff_collapsed"]

# 3. Get Satterthwaite degrees of freedom and define t-critical
df_b1 <- summary(model)$coefficients["ICPS_early_OCC_diff_collapsed", "df"]
t_crit <- qt(0.975, df = df_b1)

# 4. Define quadratic coefficients (A*X^2 + B*X + C = 0)
A <- b1^2 - (t_crit^2 * var_b1)
B <- 2 * b0 * b1 - (2 * t_crit^2 * cov_b0_b1)
C <- b0^2 - (t_crit^2 * var_b0)

# 5. Solve for roots
discriminant <- B^2 - (4 * A * C)

if (discriminant < 0) {
  cat("Confidence bands do not cross zero. The effect is everywhere or nowhere significant.\n")
} else {
  jn_lower <- (-B - sqrt(discriminant)) / (2 * A)
  jn_upper <- (-B + sqrt(discriminant)) / (2 * A)
  cat("Johnson-Neyman boundaries at X =", jn_lower, "and", jn_upper, "\n")
}

# 6. Generate plotting dataframe
x_seq <- x_seq <- seq(-3, 3, length.out = 100)

y_hat <- b0 + b1 * x_seq
se_y <- sqrt(var_b0 + (x_seq^2 * var_b1) + (2 * x_seq * cov_b0_b1))

ci_lower <- y_hat - (t_crit * se_y)
ci_upper <- y_hat + (t_crit * se_y)

plot_df <- data.frame(ICPS = x_seq, predicted = y_hat, ci_lower = ci_lower, ci_upper = ci_upper)

# 7. Plot the Johnson-Neyman interval
jn_plot <- ggplot(plot_df, aes(x = ICPS, y = predicted)) +
  geom_ribbon(aes(ymin = ci_lower, ymax = ci_upper), fill = "blue", alpha = 0.15) +
  geom_line(color = "blue", linewidth = 1) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "black", linewidth = 0.8) +
  
  # JN boundaries
  geom_vline(xintercept = jn_lower, linetype = "dotted", color = "red", linewidth = 1) +
  geom_vline(xintercept = jn_upper, linetype = "dotted", color = "red", linewidth = 1) +
  
  # Shaded significance regions 
  annotate("rect", xmin = -Inf, xmax = jn_lower, ymin = -Inf, ymax = Inf, alpha = 0.1, fill = "green") +
  annotate("rect", xmin = jn_upper, xmax = Inf, ymin = -Inf, ymax = Inf, alpha = 0.1, fill = "green") +
  
  theme_classic() +
  labs(x = "Midfrontal-Posterolateral ICPS (Error - Correct)",
       y = "Predicted sda/rd (Post-Error - Post-Correct)") +
  coord_cartesian(ylim = c(min(ci_lower), max(ci_upper)), expand = FALSE) + theme(aspect.ratio = .5)

print(jn_plot)

# Add this to your existing ggplot code


In [ ]:
# Install if necessary: install.packages("ggeffects")
library(ggeffects)

# 1. Extract adjusted predictions and confidence intervals from your specific model.
# This mathematically isolates ICPS_early_OCC_diff_collapsed while holding age, 
# social condition, sex, peer mode, and random effects constant.
eff <- ggpredict(model, terms = "ICPS_early_OCC_diff_collapsed")

# 2. Setup your original labels and PNG export
label <- "ratio_reversed_revised"
plot_label <- bquote(atop("Attentional Control", sd[a] / r[d] ~ "(Post-error - Post-correct)"))

png(file=sprintf("%s/%s_stats.png", pic_path, label), width=9, height=5, units="in", res=600)

# 3. Build the plot
g <- ggplot(data = tf_data, aes(x = ICPS_early_OCC_diff_collapsed, y = reversed_ratio_diff))

g + geom_point() + 
  
  # Replace geom_smooth() with the mathematically correct model predictions
  geom_ribbon(data = eff, aes(x = x, ymin = conf.low, ymax = conf.high), 
              inherit.aes = FALSE, fill = "grey70", alpha = 0.4) +
  geom_line(data = eff, aes(x = x, y = predicted), 
            inherit.aes = FALSE, color = "blue", linewidth = 1) +
  
  # Your exact original formatting
  labs(y = plot_label, x = "Midfrontal-Posterolateral ICPS (Error - Correct)") +
  theme_minimal() +
  theme(
    text = element_text(size = 14),
    axis.title = element_text(size = 16),
    axis.text = element_text(size = 14),
    plot.title = element_text(size = 18, face = "bold")
  ) +
  scale_y_continuous(breaks = seq(-1, 1, by = 1)) +
  scale_x_continuous(breaks = seq(-5, 5, by = 1))

dev.off()

In [ ]:
plot_df

In [ ]:
tf_data <- tf_data %>%
  filter(abs(scale(PE_diff_accuracy)) <= 3)

In [ ]:
tf_data$accuracy_score_post_trial <- scale(tf_data$accuracy_score_post_trial)
tf_data$PE_diff_accuracy <- scale(tf_data$PE_diff_accuracy)

In [ ]:
library(lme4)
library(lmerTest)

# Variables to model
vars <- c(
  # "rt_con_log_post_trial",
  # "rt_incon_log_post_trial",
  "rt_log_post_trial",
    "rt_interference_post_trial",
    "acc_interference_post_trial",
  "accuracy_score_post_trial",
  "reversed_ratio_post_trial",
  "a_post_trial"
    # "power_early"
)

# Fit models and store in a named list
models <- lapply(vars, function(v) {
  f <- as.formula(paste(v, "~ age_m * soc * acc + sex + dp_inperson + (1 | sub)"))
  lmer(f, data = tf_data)
})
names(models) <- vars

# View summaries
lapply(models, summary)

In [ ]:
summary(lmer(PE_diff_accuracy ~ age_m * soc + sex + dp_inperson + (1 | sub),
             data = subset(tf_data, tf_data$acc == 'Correct')))

In [ ]:
subset(tf_data, tf_data$acc == 'Correct')

In [ ]:
library(emmeans)
library(ggeffects)
library(ggplot2)
library(lmerTest)

# Function to test and plot ONLY significant interactions (p < 0.05)
analyze_sig_interactions <- function(model, model_name) {
  cat("\n======================================================\n")
  cat("MODEL:", model_name, "\n")
  cat("======================================================\n")
  
  # Run ANOVA to get F-tests and p-values for all terms
  aov_res <- anova(model)
  
  # Helper to check if a specific term is significant
  is_sig <- function(term) {
    if (term %in% rownames(aov_res)) {
      return(aov_res[term, "Pr(>F)"] < 0.05)
    }
    return(FALSE)
  }
  
  any_sig <- FALSE
  
  # 1. Check 2-Way: soc x acc
  if (is_sig("soc:acc")) {
    any_sig <- TRUE
    cat("\n--- SIGNIFICANT: soc x acc (FDR Corrected) ---\n")
    emm_soc_acc <- emmeans(model, ~ soc * acc)
    print(pairs(emm_soc_acc, simple = "each", adjust = "fdr"))
    
    p1 <- plot(ggpredict(model, terms = c("soc", "acc"))) +
      labs(title = paste("Soc x Acc on", model_name), x = "Condition", y = "Predicted Value", color = "Accuracy") +
      theme_minimal()
    print(p1)
  }
  
  # 2. Check 2-Way: age_m x soc
  if (is_sig("age_m:soc")) {
    any_sig <- TRUE
    cat("\n--- SIGNIFICANT: age_m x soc (FDR Corrected) ---\n")
    trends_age_soc <- emtrends(model, ~ soc, var = "age_m")
    print(test(trends_age_soc, adjust = "fdr"))
    print(pairs(trends_age_soc, adjust = "fdr"))
    
    p2 <- plot(ggpredict(model, terms = c("age_m", "soc"))) +
      labs(title = paste("Age x Soc on", model_name), x = "Age (Z-scored)", y = "Predicted Value", color = "Condition") +
      theme_minimal()
    print(p2)
  }
  
  # 3. Check 2-Way: age_m x acc
  if (is_sig("age_m:acc")) {
    any_sig <- TRUE
    cat("\n--- SIGNIFICANT: age_m x acc (FDR Corrected) ---\n")
    trends_age_acc <- emtrends(model, ~ acc, var = "age_m")
    print(test(trends_age_acc, adjust = "fdr"))
    print(pairs(trends_age_acc, adjust = "fdr"))
    
    p3 <- plot(ggpredict(model, terms = c("age_m", "acc"))) +
      labs(title = paste("Age x Acc on", model_name), x = "Age (Z-scored)", y = "Predicted Value", color = "Accuracy") +
      theme_minimal()
    print(p3)
  }
  
  # 4. Check 3-Way: age_m x soc x acc
  if (is_sig("age_m:soc:acc")) {
    any_sig <- TRUE
    cat("\n--- SIGNIFICANT 3-Way: age_m x soc x acc (FDR Corrected) ---\n")
    trends_3way <- emtrends(model, ~ soc * acc, var = "age_m")
    print(test(trends_3way, adjust = "fdr"))
    print(pairs(trends_3way, simple = "each", adjust = "fdr"))
    
    p4 <- plot(ggpredict(model, terms = c("age_m", "soc", "acc"))) +
      labs(title = paste("Age x Soc x Acc on", model_name), x = "Age (Z-scored)", y = "Predicted Value", color = "Condition") +
      theme_minimal() +
      theme(strip.text = element_text(size = 12, face = "bold"))
    print(p4)
  }
  
  if (!any_sig) {
    cat("\nNo significant interactions (p < 0.05) found for this model.\n")
  }
}

# Run the updated function
invisible(mapply(analyze_sig_interactions, models, names(models)))

In [ ]:
library(dplyr)
library(tidyr)

# 1. Create a summary dataframe with the computed difference scores
df_diff_acc <- tf_data %>%
  # Group by subject, social condition, and previous accuracy
  group_by(sub, soc, acc) %>%
  # Calculate mean post-trial accuracy, removing any NAs
  summarise(mean_post_acc = mean(accuracy_score_post_trial, na.rm = TRUE), .groups = "drop") %>%
  # Pivot to wide format: creates columns named 'acc_Error' and 'acc_Correct'
  pivot_wider(names_from = acc, values_from = mean_post_acc, names_prefix = "acc_") %>%
  # Calculate the difference: Post-Error (acc=0) minus Post-Correct (acc=1)
  mutate(PE_diff_accuracy = acc_Error - acc_Correct)

# View the exploratory summary dataframe
head(df_diff_acc)

# 2. (Optional) Merge this difference score back into your long-format data
# This assigns the exact same PE_diff_accuracy value to every trial for a given sub & soc
tf_data <- tf_data %>%
  left_join(
    df_diff_acc %>% select(sub, soc, PE_diff_accuracy),
    by = c("sub", "soc")
  )

In [ ]:
hist(tf_data$pes)

In [ ]:
hist(tf_data$PE_diff_accuracy)

# Perceived social distress analyses

In [ ]:
# # rm(list = ls())
# session = "s1_r1"
# analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
# source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
# source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
# pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
# tf_data_old <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/paper_revision/*distress*.csv", analysis_path, session)))
# table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/", analysis_path)
# if (!dir.exists(table_output_path)) {
#   dir.create(table_output_path, recursive = TRUE)
# }

In [ ]:
# tf_data[, c('sub', 'soc', 'acc', 'posttaske_i4_s1_r1_nonsoc')]

In [ ]:
# rm(list = ls())
session = "s1_r1"
analysis_path = "/home/data/NDClab/analyses/thrive-theta-ddm/"
source(sprintf("%s/code/statistics/lmer_export_apa.R", analysis_path))
source(sprintf("%s/code/statistics/find_newest_file.R", analysis_path))
pic_path <- (sprintf("%s/derivatives/figs/%s", analysis_path, session))
tf_data <- read.csv(find_newest_file(sprintf("%s/derivatives/csv/%s/paper_revision/*posttask_raw*.csv", analysis_path, session)))
table_output_path = sprintf("%s/derivatives/statistics/full_sample/ms/paper_revision/", analysis_path)
if (!dir.exists(table_output_path)) {
  dir.create(table_output_path, recursive = TRUE)
}

In [ ]:
tf_data <- tf_data %>%
  mutate(
    # Factor 1 Sums (Items 4, 5, 7)
    f1_soc = posttaske_i4_s1_r1_soc + posttaske_i5_s1_r1_soc + posttaske_i7_s1_r1_soc,
    f1_nonsoc = posttaske_i4_s1_r1_nonsoc + posttaske_i5_s1_r1_nonsoc + posttaske_i7_s1_r1_nonsoc,
    
    # Factor 2 Sums (Items 2 and 6)
    # f2_soc = posttaske_i2_s1_r1_soc + posttaske_i6_s1_r1_soc,
    # f2_nonsoc = posttaske_i2_s1_r1_nonsoc + posttaske_i6_s1_r1_nonsoc
  ) %>%
  # Drop all raw survey items to prevent dataframe clutter
  # select(-starts_with("posttaske_i")) %>%
  # Pivot the custom sum scores (and any other matched condition columns)
  pivot_longer(
    cols = matches("_soc$|_nonsoc$"),
    names_to = c(".value", "condition"),
    names_pattern = "(.*)_(soc|nonsoc)$"
  ) %>%
  # Convert condition suffix to your binary indicator
  mutate(soc = ifelse(condition == "soc", 1, 0)) %>%
  select(-condition)

# 1. Convert variables to factors
tf_data$sub <- as.factor(tf_data$sub)
tf_data$soc <- as.factor(tf_data$soc)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

# 2. Revalue 2-level factors for interpretability (if not already run in your environment
tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))

# 3. Apply sum contrast coding for factors (crucial for interpreting interactions)
contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))
# For Category (3 levels), use contr.sum dynamically based on the number of levels
# contrasts(tf_data$Category) <- contr.sum(length(levels(tf_data$Category)))

# 4. Z-score continuous predictors
tf_data$age_m <- scale(tf_data$age_m)
tf_data$f1 <- scale(tf_data$f1)
# tf_data$f2 <- scale(tf_data$f2)
# tf_data$f1_score <- scale(tf_data$f1_score)
# tf_data$f2_score <- scale(tf_data$f2_score)

In [ ]:
# 1. Define the dependent variables you want to iterate over
outcomes <- c(
  "f1"
)

# 2. Define the static right-hand side of your model formula
predictors <- "soc * age_m + sex + dp_inperson + (1 | sub)"

# 3. Initialize an empty list to store the fitted models
models_list <- list()

# 4. Iterate through each outcome
for (outcome in outcomes) {
  # Dynamically build the formula
  f <- as.formula(paste(outcome, "~", predictors))
  
  # Fit the linear mixed-effects model
  cat(sprintf("Fitting model for %s...\n", outcome))
  model <- lmer(f, data = tf_data)
  print(summary(model))
  
  # Store the model in the list using the outcome name as the key
  models_list[[outcome]] <- model
  
  # Automatically export the APA table
  label <- paste0(outcome, "_ms")
  lmer_export_apa(model = model, path = sprintf("%s/%s.docx", table_output_path, label), print_formula=TRUE, formula=f)
}


In [ ]:
# Load required libraries
if(!require(psych)) install.packages("psych")
if(!require(dplyr)) install.packages("dplyr")
library(psych)
library(dplyr)

# 1. Isolate the survey items
soc_items <- tf_data %>% select(ends_with("_soc"))
nonsoc_items <- tf_data %>% select(ends_with("_nonsoc"))
soc_items <- soc_items %>% select(-posttaske_i10_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i10_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i2_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i2_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i3_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i3_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i9_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i9_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i8_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i8_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i6_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i6_s1_r1_nonsoc)

# 2. Check data suitability (Kaiser-Meyer-Olkin & Bartlett's Test)
KMO(soc_items)
cortest.bartlett(soc_items)

KMO(nonsoc_items)
cortest.bartlett(nonsoc_items)

# 3. Determine the optimal number of factors via Parallel Analysis (Horn, 1965)
fa.parallel(soc_items, fm = "minres", fa = "fa", main = "Parallel Analysis - Social")
fa.parallel(nonsoc_items, fm = "minres", fa = "fa", main = "Parallel Analysis - Nonsocial")

# 4. Run the Exploratory Factor Analysis (EFA)
# Note: Update 'nfactors' below based on the fa.parallel() recommendations
efa_soc <- fa(soc_items, nfactors = 1, rotate = "oblimin", fm = "minres")
efa_nonsoc <- fa(nonsoc_items, nfactors = 1, rotate = "oblimin", fm = "minres")

# 5. Display results, suppressing cross-loadings below 0.3
print(efa_soc, cut = 0.3, sort = TRUE)
print(efa_nonsoc, cut = 0.3, sort = TRUE)

# 6. Test Factorial Invariance
# Calculates Tucker's congruence coefficient between the two models
factor.congruence(efa_soc, efa_nonsoc)

In [ ]:
psych::omega(final_3_items, nfactors = 1)

In [ ]:
# # Factor 1 EFA
# efa_f1 <- fa(tf_data %>% select(posttaske_i4_s1_r1, posttaske_i5_s1_r1, posttaske_i7_s1_r1), 
#              nfactors = 1, fm = "minres", scores = "regression")
# tf_data$f1_score <- as.numeric(efa_f1$scores)

# # Factor 2 EFA
# efa_f2 <- fa(tf_data %>% select(posttaske_i2_s1_r1, posttaske_i6_s1_r1), 
#              nfactors = 1, fm = "minres", scores = "regression")
# tf_data$f2_score <- as.numeric(efa_f2$scores)

In [ ]:
library(dplyr)

tf_data %>%
  group_by(soc) %>%
  summarize(
    # Factor 1 (Anxiety/Worry) raw means
    mean_i4 = mean(posttaske_i4_s1_r1, na.rm = TRUE),
    mean_i5 = mean(posttaske_i5_s1_r1, na.rm = TRUE),
    mean_i7 = mean(posttaske_i7_s1_r1, na.rm = TRUE),
    
    # Factor 2 (Calm/Confidence) raw means
    mean_i2 = mean(posttaske_i2_s1_r1, na.rm = TRUE),
    mean_i6 = mean(posttaske_i6_s1_r1, na.rm = TRUE)
  )

In [ ]:
# library(tidyr)
# library(dplyr)

# tf_data <- tf_data %>%
#   # 1. Create simple sum scores for the robust items
#   mutate(
#     f1_soc = posttaske_i4_s1_r1_soc + posttaske_i5_s1_r1_soc + posttaske_i7_s1_r1_soc,
#     f1_nonsoc = posttaske_i4_s1_r1_nonsoc + posttaske_i5_s1_r1_nonsoc + posttaske_i7_s1_r1_nonsoc
#   ) %>%
#   # 2. Drop the unstable items (2 and 6)
#   select(-contains("_i2_"), -contains("_i6_")) %>%
#   # 3. Pivot to long format
#   pivot_longer(
#     cols = matches("_soc$|_nonsoc$"),
#     names_to = c(".value", "condition"),
#     names_pattern = "(.*)_(soc|nonsoc)$"
#   ) %>%
#   # 4. Convert condition to binary 1/0
#   mutate(soc = ifelse(condition == "soc", 1, 0)) %>%
#   select(-condition)

In [ ]:
library(dplyr)

# Extract the relevant columns (using 'sub' based on your notebook's naming convention)
# and ensure they are sorted identically for fair comparison
df_new <- tf_data %>% select(sub, soc, f1) %>% arrange(sub, soc)
df_old <- tf_data_old %>% select(sub, soc, f1) %>% arrange(sub, soc)

# 1. Compare observation counts
cat("Observations in tf_data:", nrow(df_new), "\n")
cat("Observations in tf_data_old:", nrow(df_old), "\n\n")

if (nrow(df_new) != nrow(df_old)) {
  cat("WARNING: Observation counts do not match.\n\n")
} else {
  cat("Observation counts match.\n\n")
}

# 2. Find exact mismatches for sub, soc, f1 combinations
# Rows in the new dataframe not present (or different) in the old one
in_new_not_old <- setdiff(df_new, df_old)

# Rows in the old dataframe not present (or different) in the new one
in_old_not_new <- setdiff(df_old, df_new)

# 3. Output the results
if (nrow(in_new_not_old) == 0 && nrow(in_old_not_new) == 0) {
  cat("Exact match confirmed: All sub, soc, and f1 combinations are identical.\n")
} else {
  cat("--- DIFFERENCES DETECTED ---\n")
  
  if (nrow(in_new_not_old) > 0) {
    cat("\nCombinations present in tf_data but NOT in tf_data_old:\n")
    print(in_new_not_old)
  }
  
  if (nrow(in_old_not_new) > 0) {
    cat("\nCombinations present in tf_data_old but NOT in tf_data:\n")
    print(in_old_not_new)
  }
}

In [ ]:
tf_data_old

In [ ]:
soc_items <- tf_data %>% select(ends_with("_soc"))
nonsoc_items <- tf_data %>% select(ends_with("_nonsoc"))
soc_items <- soc_items %>% select(-posttaske_i10_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i10_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i2_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i2_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i3_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i3_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i9_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i9_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i8_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i8_s1_r1_nonsoc)
soc_items <- soc_items %>% select(-posttaske_i6_s1_r1_soc)
nonsoc_items <- nonsoc_items %>% select(-posttaske_i6_s1_r1_nonsoc)

print(psych::omega(soc_items, nfactors = 1))
print(psych::omega(nonsoc_items, nfactors = 1))

In [ ]:
# Corrected measurement model syntax matching your dataframe's exact column names
cfa_model <- '
  F1 =~ posttaske_i4_s1_r1 + posttaske_i5_s1_r1 + posttaske_i7_s1_r1
  F2 =~ a*posttaske_i2_s1_r1 + a*posttaske_i6_s1_r1
'

# 1. Configural Invariance
fit_configural <- cfa(cfa_model, 
                      data = tf_data, 
                      group = "soc")

# 2. Metric Invariance
fit_metric <- cfa(cfa_model, 
                  data = tf_data, 
                  group = "soc", 
                  group.equal = c("loadings"))

# Compare the models
lavTestLRT(fit_configural, fit_metric)

In [ ]:
# Extract fit indices for both models
round(fitmeasures(fit_configural, c("cfi", "rmsea")), 3)
round(fitmeasures(fit_metric, c("cfi", "rmsea")), 3)

In [ ]:
fa(soc_items, nfactors = 3)

In [ ]:
# Anticipated next step: Cleaned extraction
clean_soc_items <- tf_data %>% select(ends_with("_soc"), -contains("_i9_"), -contains("_i10_"))
clean_nonsoc_items <- tf_data %>% select(ends_with("_nonsoc"), -contains("_i9_"), -contains("_i10_"))

# Re-verify parallel analysis (should now cleanly recommend 2 factors for both)
fa.parallel(clean_soc_items, fm = "minres", fa = "fa")

# Rerun final EFA and extract scores
efa_soc_clean <- fa(clean_soc_items, nfactors = 2, rotate = "oblimin", fm = "minres")
efa_nonsoc_clean <- fa(clean_nonsoc_items, nfactors = 2, rotate = "oblimin", fm = "minres")

factor.congruence(efa_soc_clean, efa_nonsoc_clean)

In [ ]:
# 1. Isolate the robust items (4, 5, and 7) based on the prior invariance testing
soc_robust <- tf_data %>% select(posttaske_i4_s1_r1_soc, posttaske_i5_s1_r1_soc, posttaske_i7_s1_r1_soc)
nonsoc_robust <- tf_data %>% select(posttaske_i4_s1_r1_nonsoc, posttaske_i5_s1_r1_nonsoc, posttaske_i7_s1_r1_nonsoc)

# 2. Extract 1-factor regression scores independently to preserve condition-specific variance
efa_soc_1f <- fa(soc_robust, nfactors = 1, fm = "minres")
efa_nonsoc_1f <- fa(nonsoc_robust, nfactors = 1, fm = "minres")

# 3. Append the scores to the wide dataframe
tf_data$f1_soc <- as.numeric(efa_soc_1f$scores)
tf_data$f1_nonsoc <- as.numeric(efa_nonsoc_1f$scores)

# 4. Pivot to long format
tf_data <- tf_data %>%
  # Drop the unstable items (2 and 6) to keep the dataframe clean
  select(-contains("_i2_"), -contains("_i6_")) %>%
  # Pivot all columns ending in _soc or _nonsoc
  pivot_longer(
    cols = matches("_soc$|_nonsoc$"),
    names_to = c(".value", "condition"),
    names_pattern = "(.*)_(soc|nonsoc)$"
  ) %>%
  # Convert the string condition column to a binary 1/0 column
  mutate(soc = ifelse(condition == "soc", 1, 0)) %>%
  # Drop the redundant character column
  select(-condition)

In [ ]:
tf_data

In [ ]:
# tf_data

In [ ]:
contrasts(tf_data$soc)

In [ ]:
# Model 1: Main effects of soc, age_m, sex, and dp_inperson on f1
model_f1 <- lmer(f1 ~ soc + age_m + sex + dp_inperson + (1 | sub), data = tf_data)
summary(model_f1)

# Model 2: Main effects of soc, age_m, sex, and dp_inperson on f2
model_f2 <- lmer(f2 ~ soc + age_m + sex + dp_inperson + (1 | sub), data = tf_data)
summary(model_f2)

In [ ]:
# Model 1: Main effects of soc, age_m, sex, and dp_inperson on f1
model_f1 <- lmer(f1 ~ soc + age_m + sex + dp_inperson + (1 | sub), data = tf_data)
summary(model_f1)

# # Model 2: Main effects of soc, age_m, sex, and dp_inperson on f2
model_f2 <- lmer(f2 ~ soc + age_m + sex + dp_inperson + (1 | sub), data = tf_data)
summary(model_f2)

In [ ]:
id_cols <- c('sub', 'age_m', 'sex', 'acc', 'soc', 'dp_inperson', 'first_soc')
data_cols <- c(
    'power_early',
    'ITPS_early',
    'ICPS_early_DLPFC_collapsed',
    'ICPS_early_MOTOR_collapsed',
    'ICPS_early_OCC_collapsed'
)
tf_data <- tf_data[, c(id_cols, data_cols)]
tf_data <- tf_data %>%
  filter(!if_all(c(data_cols), is.na))

tf_data$soc <- as.factor(tf_data$soc)
tf_data$sub <- as.factor(tf_data$sub)
tf_data$sex <- as.factor(tf_data$sex)
tf_data$acc <- as.factor(tf_data$acc)
tf_data$first_soc <- as.factor(tf_data$first_soc)
# tf_data$task_num <- as.factor(tf_data$task_num)
tf_data$dp_inperson <- as.factor(tf_data$dp_inperson)

tf_data[c("age_m", data_cols)] <- as.data.frame(scale(tf_data[c("age_m", data_cols)]))

tf_data$soc <- revalue(tf_data$soc, c("0" = "NS", "1" = "S"))
tf_data$sex <- revalue(tf_data$sex, c("1" = "M", "2" = "F"))
tf_data$acc <- revalue(tf_data$acc, c("0" = "Error", "1" = "Correct"))

contrasts(tf_data$soc) <- rev(contr.sum(2))
contrasts(tf_data$acc) <- rev(contr.sum(2))
contrasts(tf_data$first_soc) <- rev(contr.sum(2))
contrasts(tf_data$sex) <- rev(contr.sum(2))
# contrasts(tf_data$task_num) <- rev(contr.sum(2))
contrasts(tf_data$dp_inperson) <- rev(contr.sum(2))